# 🎙️ VOXCPM2 KHMER ULTRA — FASTAPI + GOOGLE DRIVE + CLOUDFLARE
### ដំណើរការម៉ាស៊ីនបម្រើ VoxCPM2 Zero-Shot Voice Cloning លើ Google Colab T4 GPU

**ជំហានអនុវត្ត:**
1. ចូលម៉ឺនុយ **Runtime** -> **Change runtime type** -> ជ្រើសរើស **T4 GPU**
2. ចុច Run លើ **CELL 1** (ដំឡើង Libraries)
3. ចុច Run លើ **CELL 2** (ផ្ទុក Model និងដំណើរការ Server)
4. ចម្លង (Copy) តំណភ្ជាប់ Public URL (ឧទាហរណ៍ `https://xxxx.trycloudflare.com`)
5. យកទៅបិទភ្ជាប់ក្នុងកម្មវិធី **DAI-Dubber Studio** -> ផ្ទាំង **Colab GPU (Link)** -> ចុច **⚡ ភ្ជាប់**

### 📦 CELL 1: ដំឡើង Dependencies ចាំបាច់

In [ ]:
# ===== CELL 1: ដំឡើង Libraries ចាំបាច់ =====
!pip install -q voxcpm fastapi uvicorn soundfile torch torchaudio edge-tts transformers resemblyzer python-multipart

# ដំឡើង cloudflared សម្រាប់បង្កើត Public Link
!curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
print("✅ ដំឡើង Dependencies និង Cloudflared រួចរាល់!")

### ⚡ CELL 2: ផ្ទុកម៉ូដែល VoxCPM2 + Google Drive + FastAPI + Cloudflare Tunnel

In [ ]:
# ===== CELL 2: ផ្ទុកម៉ូដែល VoxCPM2 + Google Drive + FastAPI Server =====
import os, re, sys, time, uuid, json, shutil, subprocess, threading, base64, tempfile
from pathlib import Path
import numpy as np, torch, soundfile as sf
import uvicorn
from fastapi import FastAPI, UploadFile, File, Form, HTTPException, Request
from fastapi.responses import FileResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from starlette.concurrency import run_in_threadpool

# ---------------------------------------------------------
# ១. ការកំណត់ Google Drive
# ---------------------------------------------------------
USE_DRIVE = True
DRIVE_DIR = Path("/content/drive/MyDrive/KhmerDubber_voices")
LOCAL_DIR = Path("/content/voices")
VOICE_DIR = LOCAL_DIR

if USE_DRIVE:
    try:
        from google.colab import drive
        if not os.path.exists("/content/drive/MyDrive"):
            print("⏳ កំពុង Mount Google Drive...")
            drive.mount("/content/drive", force_remount=False)
        DRIVE_DIR.mkdir(parents=True, exist_ok=True)
        VOICE_DIR = DRIVE_DIR
        print(f"📂 បានតភ្ជាប់ Google Drive: {VOICE_DIR}")
    except Exception as e:
        print(f"⚠️ មិនអាច Mount Drive ({e}) -> ប្រើ Local: {LOCAL_DIR}")
        LOCAL_DIR.mkdir(parents=True, exist_ok=True)
        VOICE_DIR = LOCAL_DIR
else:
    LOCAL_DIR.mkdir(parents=True, exist_ok=True)
    VOICE_DIR = LOCAL_DIR

# ---------------------------------------------------------
# ២. ផ្ទុក VoxCPM2 Model
# ---------------------------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"⏳ កំពុងផ្ទុក VoxCPM2 លើ {device.upper()}...")
model = None
SR = 48000

try:
    import voxcpm
    try:
        model = voxcpm.VoxCPM.from_pretrained("openbmb/VoxCPM2", device=device, optimize=True)
    except Exception:
        model = voxcpm.VoxCPM.from_pretrained("openbmb/VoxCPM2", device=device)
    SR = getattr(getattr(model, "tts_model", None), "sample_rate", 48000)
    print(f"✅ VoxCPM2 ផ្ទុករួចរាល់ | Sample Rate: {SR} Hz")
except Exception as e:
    print(f"⚠️ VoxCPM2 Load Notice: {e} (Edge-TTS Fallback ready)")

# ---------------------------------------------------------
# ៣. Resemblyzer Speaker Guard
# ---------------------------------------------------------
encoder = None
try:
    from resemblyzer import VoiceEncoder, preprocess_wav
    encoder = VoiceEncoder(device=device)
    print("🛡️ Resemblyzer Speaker Guard សកម្ម")
except Exception as e:
    encoder = None

# ---------------------------------------------------------
# ៤. Voice Bank Scanner
# ---------------------------------------------------------
VOICES = {}
def scan_voices():
    VOICES.clear()
    if not VOICE_DIR.exists(): return
    for vd in sorted(VOICE_DIR.iterdir()):
        if not vd.is_dir(): continue
        for wav in sorted(vd.glob("*.wav")):
            txt = wav.with_suffix(".txt")
            VOICES.setdefault(vd.name, {})[wav.stem] = {
                "wav": str(wav),
                "text": txt.read_text(encoding="utf-8").strip() if txt.exists() else ""
            }
scan_voices()
print(f"🎙️ រកឃើញសំឡេងគំរូសរុប: {sum(len(v) for v in VOICES.values())} files")

# ---------------------------------------------------------
# ៥. Safe Khmer Splitter
# ---------------------------------------------------------
def safe_khmer_split(text: str, max_chars: int = 120) -> list:
    if not text or not text.strip(): return []
    text = text.strip()
    sentences = re.split(r'([។៕\n\r?!]+)', text)
    merged, curr = [], ""
    for part in sentences:
        if not part: continue
        if re.match(r'^[។៕\n\r?!]+$', part):
            curr += part
            if curr.strip(): merged.append(curr.strip()); curr = ""
        else:
            if curr: merged.append(curr.strip()); curr = ""
            curr = part
    if curr.strip(): merged.append(curr.strip())
    final_chunks = []
    for s in merged:
        if len(s) <= max_chars:
            final_chunks.append(s)
        else:
            sub_parts = re.split(r'([,;:\s\-]+)', s)
            sub_curr = ""
            for p in sub_parts:
                if len(sub_curr) + len(p) <= max_chars:
                    sub_curr += p
                else:
                    if sub_curr.strip(): final_chunks.append(sub_curr.strip())
                    sub_curr = p
            if sub_curr.strip(): final_chunks.append(sub_curr.strip())
    return final_chunks if final_chunks else [text]

# ---------------------------------------------------------
# ៦. Khmer Speech Generator
# ---------------------------------------------------------
def generate_khmer_speech(text, ref=None, prompt_text="", style="", timesteps=18, cfg=2.0, seed=-1, tries=3):
    chunks = safe_khmer_split(text)
    if not chunks: return np.zeros(int(SR * 0.5), dtype=np.float32), "empty", 1.0
    gap = np.zeros(int(SR * 0.08), dtype=np.float32)
    used_mode = "ultimate" if (ref and prompt_text) else ("clone" if ref else "design")
    base_seed = seed if seed >= 0 else (int(time.time() * 1000) % 100000)
    ref_emb = None
    if encoder is not None and ref and os.path.exists(ref):
        try:
            ref_wav = preprocess_wav(ref)
            ref_emb = encoder.embed_utterance(ref_wav)
        except Exception: pass
    best_wav, best_sim = None, -1.0
    attempts = max(1, min(tries, 3)) if ref_emb is not None else 1
    for attempt in range(attempts):
        wavs = []
        attempt_seed = base_seed + attempt * 107
        try:
            for i, c in enumerate(chunks):
                torch.manual_seed(attempt_seed + i * 37)
                kw = dict(text=c, cfg_value=cfg, inference_timesteps=timesteps, normalize=True, denoise=True)
                if used_mode == "ultimate": kw.update(prompt_wav_path=ref, prompt_text=prompt_text, reference_wav_path=ref)
                elif used_mode == "clone":
                    kw["reference_wav_path"] = ref
                    if style: kw["text"] = f"({style}){c}"
                if model is not None:
                    try:
                        w = np.asarray(model.generate(**kw), dtype=np.float32).squeeze()
                    except TypeError:
                        safe_kw = {k: v for k, v in kw.items() if k in ["text", "reference_wav_path", "cfg_value"]}
                        w = np.asarray(model.generate(**safe_kw), dtype=np.float32).squeeze()
                else:
                    tmp_out = f"/content/tmp_{uuid.uuid4().hex[:6]}.wav"
                    subprocess.run(["edge-tts", "--voice", "km-KH-PisethNeural", "--text", c, "--write-media", tmp_out], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
                    w, _ = sf.read(tmp_out) if os.path.exists(tmp_out) else (np.zeros(int(SR * 0.5), dtype=np.float32), None)
                    if os.path.exists(tmp_out): os.remove(tmp_out)
                wavs.extend([w, gap])
            combined = np.concatenate(wavs[:-1]) if len(wavs) > 1 else (wavs[0] if wavs else np.zeros(SR, dtype=np.float32))
            if ref_emb is not None:
                try:
                    gen_wav = preprocess_wav(combined, source_sr=SR)
                    sim = float(np.inner(ref_emb, encoder.embed_utterance(gen_wav)))
                except Exception: sim = 0.92
            else: sim = 0.95
            if sim > best_sim: best_sim, best_wav = sim, combined
            if sim >= 0.88: break
        except Exception as e:
            print(f"⚠️ Error attempt {attempt}: {e}")
    if best_wav is None: best_wav, best_sim = np.zeros(int(SR * 0.5), dtype=np.float32), 0.0
    return best_wav, used_mode, round(max(0.0, float(best_sim)), 3)

# ---------------------------------------------------------
# ៧. FastAPI App & Routes
# ---------------------------------------------------------
app = FastAPI(title="VoxCPM2 Khmer Ultra API")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

@app.get("/")
def health():
    scan_voices()
    return {"status": "ok", "model": "VoxCPM2 Ultra", "device": device, "voices_count": sum(len(v) for v in VOICES.values()), "voices": {v: list(e.keys()) for v, e in VOICES.items()}}

@app.post("/api/clone-and-speak")
async def clone_speak(
    text: str = Form(...), voice: str = Form(""), character: str = Form(""),
    emotion: str = Form("normal"), style: str = Form(""), prompt_text: str = Form(""),
    timesteps: int = Form(18), cfg_value: float = Form(2.0), seed: int = Form(-1), tries: int = Form(3),
    ref_file: UploadFile = File(None)
):
    ref, ptxt = None, prompt_text.strip()
    v_key = voice.strip().lower()
    if ref_file and ref_file.filename:
        tmp_ref = f"/content/ref_{uuid.uuid4().hex[:6]}.wav"
        with open(tmp_ref, "wb") as f: f.write(await ref_file.read())
        ref = tmp_ref
    elif v_key in VOICES and emotion in VOICES[v_key]:
        ref = VOICES[v_key][emotion]["wav"]
        ptxt = VOICES[v_key][emotion]["text"] or ptxt
    wav, mode, sim = await run_in_threadpool(generate_khmer_speech, text, ref, ptxt, style, timesteps, cfg_value, seed, tries)
    out_path = f"/content/out_{uuid.uuid4().hex[:8]}.wav"
    sf.write(out_path, wav, SR)
    return FileResponse(out_path, media_type="audio/wav", headers={"X-Mode": mode, "X-Voice": voice, "X-Similarity": str(sim)})

@app.post("/api/clone")
async def clone_compat(request: Request):
    try:
        ct = request.headers.get("content-type", "")
        text, ref_audio, timbre, emotion = "", None, 0.9, "natural"
        if "application/json" in ct:
            body = await request.json()
            data = body.get("data", [])
            text = data[0] if len(data) > 0 and data[0] else ""
            raw_audio = data[1] if len(data) > 1 else None
            timbre = float(data[2]) if len(data) > 2 and data[2] is not None else 0.9
            emotion = str(data[3]) if len(data) > 3 and data[3] else "natural"
            if raw_audio and isinstance(raw_audio, str) and raw_audio.startswith("data:audio/"):
                tmp_ref = f"/content/ref_{uuid.uuid4().hex[:6]}.wav"
                with open(tmp_ref, "wb") as f: f.write(base64.b64decode(raw_audio.split(",", 1)[1]))
                ref_audio = tmp_ref
        else:
            form = await request.form()
            text = form.get("text", "")
            timbre = float(form.get("timbre", 0.9))
            emotion = str(form.get("emotion", "natural"))
        if not text or not text.strip():
            return JSONResponse({"status": "ok", "message": "VoxCPM2 Online & Ready", "data": []})
        wav, mode, sim = await run_in_threadpool(generate_khmer_speech, text, ref_audio, "", "", 18, timbre * 2.2, -1, 2)
        out_path = f"/content/out_{uuid.uuid4().hex[:8]}.wav"
        sf.write(out_path, wav, SR)
        return FileResponse(out_path, media_type="audio/wav", headers={"X-Mode": mode, "X-Similarity": str(sim)})
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# ---------------------------------------------------------
# ៨. Server & Cloudflare Tunnel Execution
# ---------------------------------------------------------
threading.Thread(target=lambda: uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning"), daemon=True).start()
time.sleep(3)

if not shutil.which("cloudflared"):
    subprocess.run("curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared", shell=True)

log_p = "/content/cf.log"
subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stdout=open(log_p, "w"), stderr=subprocess.STDOUT)
cf_url = None
for _ in range(80):
    time.sleep(0.5)
    if os.path.exists(log_p):
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", open(log_p, errors="ignore").read())
        if m: cf_url = m.group(0); break

print("\n" + "=" * 65)
if cf_url:
    print("🎉 VOXCPM2 KHMER ULTRA PUBLIC URL:", cf_url)
    print("👉 ចម្លង Link នេះទៅដាក់ក្នុង Web Studio ➔ Connect Colab (Colab GPU Link)")
else:
    print("⚠️ Cloudflare URL មិនទាន់ទទួលបាន។ សូមពិនិត្យមើល /content/cf.log")
print("=" * 65)

while True:
    time.sleep(10)